# USAGE: PUT RAW DATA IN ../data/wifi_samples.csv, PLUG-AND-PLAY

## Preliminary processing and cleaning of raw samples

In [ ]:
# import json
import math
from io import BytesIO
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.collections import PolyCollection

CSV_PATH = Path("../data/wifi_samples.csv")
PHONE_LOCATION_LOG_PATH = Path("../data/phone_location_log.csv")
WDUTIL_RSSI_PLACEHOLDER_FILL_DBM = None
ASOF_COORDINATE_LOOKBACK = pd.Timedelta(seconds=30)
# pref corelocation first.
COORDINATE_PRIORITY = (
    "mac_corelocation", "corelocation",
    "phone_gps", "phone_gps_asof",
)
COORDINATE_LABELS = {
    "phone_gps": "phone GPS",
    "mac_corelocation": "mac CoreLocation",
    "corelocation": "legacy CoreLocation",
    "phone_gps_asof": "phone GPS asof",
}
ASOF_COORDINATE_COLUMNS = [
    "phone_gps_asof_timestamp",
    "phone_gps_asof_sample_index",
    "phone_gps_asof_latitude",
    "phone_gps_asof_longitude",
    "phone_gps_asof_altitude_m",
]
DATAFRAME_COLUMNS = [
    "measurement_set_id",
    "sample_index",
    "collector_id",
    "environment",
    "building",
    "floor",
    "waypoint_id",
    "measurement_set_timestamp_utc",
    "location_timestamp",
    "latitude",
    "longitude",
    "h_accuracy_m",
    "mac_location_timestamp",
    "mac_latitude",
    "mac_longitude",
    "mac_h_accuracy_m",
    "phone_location_timestamp",
    "phone_latitude",
    "phone_longitude",
    "phone_altitude_m",
    "selected_location_source",
    "selected_location_timestamp",
    "selected_latitude",
    "selected_longitude",
    "wdutil_rssi_dbm",
    "wdutil_rssi_effective_dbm",
    "wdutil_noise_dbm",
    "wdutil_noise_effective_dbm",
    "wdutil_tx_rate",
    "wdutil_zero_placeholder",
]
NUMERIC_COLUMNS = [
    "sample_index",
    "floor",
    "latitude",
    "longitude",
    "h_accuracy_m",
    "mac_latitude",
    "mac_longitude",
    "mac_h_accuracy_m",
    "phone_latitude",
    "phone_longitude",
    "phone_altitude_m",
    "wdutil_rssi_dbm",
    "wdutil_noise_dbm",
]


def resolve_rssi_placeholder_fill_value(wifi_df):
    '''
    NOTE USED: fill with np.nan for now
    '''

    if WDUTIL_RSSI_PLACEHOLDER_FILL_DBM is not None:
        return float(WDUTIL_RSSI_PLACEHOLDER_FILL_DBM)
    observed_rssi = wifi_df.loc[
        wifi_df["wdutil_rssi_dbm"].notna() & wifi_df["wdutil_rssi_dbm"].ne(0),
        "wdutil_rssi_dbm",
    ]
    if observed_rssi.empty:
        raise ValueError(
            "Could not infer an RSSI fill value because there are no nonzero wdutil_rssi_dbm values.")

    observed_noise = wifi_df.loc[
        wifi_df["wdutil_noise_dbm"].notna(
        ) & wifi_df["wdutil_noise_dbm"].ne(0),
        "wdutil_noise_dbm",
    ]
    return float(observed_rssi.min()), float(observed_noise.min())


def build_phone_gps_asof_dataframe(wifi_df, phone_location_log_path=PHONE_LOCATION_LOG_PATH):
    phone_gps_asof_df = pd.DataFrame(
        index=wifi_df.index, columns=ASOF_COORDINATE_COLUMNS)
    phone_location_log_path = Path(phone_location_log_path)
    if not phone_location_log_path.exists():
        return phone_gps_asof_df

    phone_log_df = pd.read_csv(
        phone_location_log_path,
        header=None,
        names=ASOF_COORDINATE_COLUMNS,
        na_values=["null", "none", "nan"],
        keep_default_na=True,
    )

    for column in phone_log_df.columns:
        if pd.api.types.is_object_dtype(phone_log_df[column]) or pd.api.types.is_string_dtype(phone_log_df[column]):
            phone_log_df[column] = phone_log_df[column].map(
                lambda value: value.strip() if isinstance(value, str) else value
            )
    phone_log_df = phone_log_df.replace(r"^\\s*$", pd.NA, regex=True)

    for column in ASOF_COORDINATE_COLUMNS[1:]:
        phone_log_df[column] = pd.to_numeric(
            phone_log_df[column], errors="coerce")

    phone_log_df["phone_gps_asof_lookup_timestamp"] = pd.to_datetime(
        phone_log_df["phone_gps_asof_timestamp"],
        format="mixed",
        utc=True,
    )
    phone_log_df = phone_log_df.dropna(subset=["phone_gps_asof_lookup_timestamp"]).sort_values(
        "phone_gps_asof_lookup_timestamp",
        kind="stable",
    )

    wifi_lookup_df = pd.DataFrame(index=wifi_df.index)
    wifi_lookup_df["__wifi_index"] = wifi_df.index
    wifi_lookup_df["phone_gps_asof_lookup_timestamp"] = pd.to_datetime(
        wifi_df["sample_timestamp_utc"].combine_first(
            wifi_df["measurement_set_timestamp_utc"]),
        format="mixed",
        utc=True,
    )
    valid_wifi_mask = wifi_lookup_df["phone_gps_asof_lookup_timestamp"].notna()
    if phone_log_df.empty or not valid_wifi_mask.any():
        return phone_gps_asof_df

    matched_asof_df = pd.merge_asof(
        wifi_lookup_df.loc[valid_wifi_mask].sort_values(
            "phone_gps_asof_lookup_timestamp",
            kind="stable",
        ),
        phone_log_df,
        on="phone_gps_asof_lookup_timestamp",
        direction="backward",
        tolerance=ASOF_COORDINATE_LOOKBACK,
    ).set_index("__wifi_index")

    phone_gps_asof_df.loc[matched_asof_df.index, ASOF_COORDINATE_COLUMNS] = matched_asof_df[
        ASOF_COORDINATE_COLUMNS
    ]
    return phone_gps_asof_df


def summarize_coordinate_sources(wifi_df):
    counts = wifi_df["selected_location_source"].value_counts(dropna=False)
    parts = []
    for source in COORDINATE_PRIORITY:
        count = int(counts.get(source, 0))
        if count:
            parts.append(f"{count} {COORDINATE_LABELS[source]}")
    missing = int(wifi_df["selected_location_source"].isna().sum())
    if missing:
        parts.append(f"{missing} missing coordinates")
    return ", ".join(parts)


def build_wifi_dataframe(csv_path=CSV_PATH):
    wifi_df = pd.read_csv(csv_path, na_values=[
                          "null", "none", "nan"], keep_default_na=True)

    for column in wifi_df.columns:
        if pd.api.types.is_object_dtype(wifi_df[column]) or pd.api.types.is_string_dtype(wifi_df[column]):
            wifi_df[column] = wifi_df[column].map(
                lambda value: value.strip() if isinstance(value, str) else value
            )
    wifi_df = wifi_df.replace(r"^\\s*$", pd.NA, regex=True)

    for column in NUMERIC_COLUMNS:
        if column in wifi_df.columns:
            wifi_df[column] = pd.to_numeric(wifi_df[column], errors="coerce")

    phone_gps_asof_df = build_phone_gps_asof_dataframe(wifi_df)

    tx_rate_mbps = pd.to_numeric(
        wifi_df["wdutil_tx_rate"].astype(
            "string").str.extract(r"([0-9.]+)", expand=False),
        errors="coerce",
    )
    wifi_df["wdutil_zero_placeholder"] = (
        wifi_df["wdutil_rssi_dbm"].eq(0) | wifi_df["wdutil_noise_dbm"].eq(0))
    # (
    #     wifi_df["wdutil_rssi_dbm"].ge(0)
    #     & wifi_df["wdutil_noise_dbm"].ge(0)
    #     & tx_rate_mbps.fillna(0).eq(0)
    # )
    # rssi_fill_value, noise_fill_value = resolve_rssi_placeholder_fill_value(wifi_df)
    wifi_df["wdutil_rssi_effective_dbm"] = wifi_df["wdutil_rssi_dbm"]
    wifi_df["wdutil_noise_effective_dbm"] = wifi_df["wdutil_noise_dbm"]
    wifi_df.loc[wifi_df["wdutil_zero_placeholder"],
                "wdutil_rssi_effective_dbm"] = np.nan
    wifi_df.loc[wifi_df["wdutil_zero_placeholder"],
                "wdutil_noise_effective_dbm"] = np.nan
    # wifi_df.attrs["wdutil_rssi_placeholder_fill_dbm"] = rssi_fill_value

    phone_available = wifi_df["phone_latitude"].notna(
    ) & wifi_df["phone_longitude"].notna()
    mac_available = wifi_df["mac_latitude"].notna(
    ) & wifi_df["mac_longitude"].notna()
    core_available = wifi_df["latitude"].notna() & wifi_df["longitude"].notna()
    phone_gps_asof_available = (
        phone_gps_asof_df["phone_gps_asof_latitude"].notna()
        & phone_gps_asof_df["phone_gps_asof_longitude"].notna()
    )

    wifi_df["selected_location_source"] = pd.Series(
        pd.NA, index=wifi_df.index, dtype="object")
    wifi_df.loc[phone_available, "selected_location_source"] = "phone_gps"
    wifi_df.loc[~phone_available & mac_available,
                "selected_location_source"] = "mac_corelocation"
    wifi_df.loc[
        ~phone_available & ~mac_available & core_available,
        "selected_location_source",
    ] = "corelocation"
    wifi_df.loc[
        ~phone_available & ~mac_available & ~core_available & phone_gps_asof_available,
        "selected_location_source",
    ] = "phone_gps_asof"

    wifi_df["selected_location_timestamp"] = (
        wifi_df["phone_location_timestamp"]
        .combine_first(wifi_df["mac_location_timestamp"])
        .combine_first(wifi_df["location_timestamp"])
        .combine_first(phone_gps_asof_df["phone_gps_asof_timestamp"])
    )
    wifi_df["selected_latitude"] = (
        wifi_df["phone_latitude"]
        .combine_first(wifi_df["mac_latitude"])
        .combine_first(wifi_df["latitude"])
        .combine_first(phone_gps_asof_df["phone_gps_asof_latitude"])
    )
    wifi_df["selected_longitude"] = (
        wifi_df["phone_longitude"]
        .combine_first(wifi_df["mac_longitude"])
        .combine_first(wifi_df["longitude"])
        .combine_first(phone_gps_asof_df["phone_gps_asof_longitude"])
    )

    return wifi_df.reindex(columns=DATAFRAME_COLUMNS)


def write_wifi_dataframe(wifi_df, output_path):
    output_path = Path(output_path)
    wifi_df.to_csv(output_path, index=False, na_rep="null")
    return output_path

## Save full raw data

In [ ]:
wifi_df = build_wifi_dataframe()
print(f"Loaded {len(wifi_df)} rows into pandas.DataFrame with {wifi_df.shape[1]} columns.")
print(
    "Coordinate priority for selected_location_* columns: "
    + " -> ".join(COORDINATE_LABELS[source] for source in COORDINATE_PRIORITY)
)
print(
    f"Using NaN as the fill value for {int(wifi_df['wdutil_zero_placeholder'].sum())} wdutil zero-placeholder rows."
)
print(f"Selected coordinate coverage: {summarize_coordinate_sources(wifi_df)}.")

write_wifi_dataframe(wifi_df, '../data/data.csv')
display(wifi_df.head(12))

## Clean data to prepare for inference

In [ ]:
# grab "interesting" columns, drop anything without location
df_clean = wifi_df[['environment', 'floor',
                    'selected_location_timestamp',
                    'selected_latitude',
                    'selected_longitude',
                    'wdutil_rssi_effective_dbm',
                    'wdutil_noise_effective_dbm',
                    'wdutil_zero_placeholder']].copy()
df_clean = df_clean.dropna(subset=['selected_latitude', 'selected_longitude'])
df_clean['wdutil_zero_placeholder'] = ~df_clean['wdutil_zero_placeholder']
df_clean.columns = ['environment', 'floor', 'datetime', 'latitude', 'longitude', 'rssi', 'noise', 'signal_measured']

df_clean['indoor'] = (df_clean['environment'] == 'indoor')
df_clean['floor'] = df_clean['floor'].fillna(0)

from datetime import timezone, timedelta

# convert to datetime and all to PDT
df_clean['datetime'] = pd.to_datetime(df_clean['datetime'], format='mixed', utc=True).dt.tz_convert("America/Los_Angeles")
df_clean['date'] = df_clean['datetime'].dt.date
df_clean['time_pdt'] = df_clean['datetime'].dt.time

# reorder columns
df_clean = df_clean[['date', 'time_pdt', 'latitude', 'longitude', 'indoor', 'floor', 'rssi', 'noise', 'signal_measured', 'datetime']]

assert df_clean.isna().sum(axis=0).max() == (~df_clean['signal_measured']).sum(), "Unhandled NAs present in data"

write_wifi_dataframe(df_clean, '../data/df_clean.csv')
display(df_clean.head(12))